# ATP Tennis Match Prediction (2000-2024)
---

## 1. Introduzione e Obiettivi

L'obiettivo di questo progetto è sviluppare un modello di Machine Learning basato due modelli di allenamento in grado di predire il vincitore di una partita di tennis professionistico ATP.
Il tennis è uno sport caratterizzato da una forte componente stocastica, ma anche da pattern storici ben definiti come gerarchie di ranking, punteggio di ELO, adattamento alle superfici, scontri diretti.

I dati utilizzati provengono dal repository pubblico di Jeff Sackmann (https://github.com/JeffSackmann/tennis_atp/tree/master). Il dataset completo copre l'arco temporale 1968-2024. Per questo progetto abbiamo scelto di iniziare dall'anno 2000 per garantire una maggiore consistenza nella raccolta statistica (es. minuti giocati, ace, doppi falli) che negli anni precedenti era spesso frammentaria.

---

## 2. Data Ingestion & Merging
I dati grezzi sono suddivisi in file CSV annuali (un file per anno). Il primo passo della pipeline è l'aggregazione di questi file in un unico DataFrame Pandas cronologicamente ordinato.
Durante la fase di ingestionrecupero dei dati, abbiamo applicato i seguenti filtri per garantire una qualità alta del dato:
- Esclusione dei tornei "Futures": I dati relativi ai tornei minori (Futures) sono stati esclusi. Questi match presentano spesso statistiche incomplete o inaffidabili e introducono un elevato livello di rumore che potrebbe confondere il modello.
- Esclusione delle Qualificazioni: Sono stati considerati solo i match dei tabelloni principali del circuito ATP e Challenger.
- Parsing delle Date: La colonna tourney_date è stata convertita immediatamente in formato datetime per permettere il futuro split cronologico per il Training e il Testing evitando il Data Leakage temporale.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

Come prima cosa sono stati sono stati scaricati i dataset da GitHub e sono stati inseriti nella cartella 'datasets'
Una volta impostato il nome del file mergiato, viene creato un dataframe temporaneo contenente tutti i dataset annuali in cui le date dei match sono state convertite nel formato datetime

In [5]:
INPUT_FOLDER = "datasets"
OUTPUT_FILENAME = "atp_matches_2000_2024_raw.csv"

print("--- UNIONE DATI ---")
    
    # Cerca file che iniziano con "atp_matches_" dentro la cartella datasets
pattern = os.path.join(INPUT_FOLDER, "atp_matches_*.csv")
all_files = glob.glob(pattern)
    

li = []
for filename in all_files:
    try:
        df_temp = pd.read_csv(filename, index_col=None, header=0)
            
        # Convertiamo subito la data (formato YYYYMMDD) in oggetto datetime
        df_temp['tourney_date'] = pd.to_datetime(df_temp['tourney_date'], format='%Y%m%d', errors='coerce')
            
        li.append(df_temp)
    except Exception as e:
        print(f"Errore caricamento {filename}: {e}")

if not li:
    print("Errore: Nessun file caricato.")
    
print("--- UNIONE COMPLETATA ---")

--- UNIONE DATI ---
--- UNIONE COMPLETATA ---


Per scrupolo il dataset viene ordinato secondo 'tourney_date' per rispettare esattamente la realtà temporale. Questo passaggioo è fondamentale perchè L'algoritmo ELO (che calcoleremo successivamente) è calcolato ricorsivamente: il punteggio di un giocatore oggi dipende dal risultato di ieri.
Ordiniamo prima per Data (tourney_date). A parità di data (es. tutti i match del primo turno di Wimbledon), usiamo l'ID del torneo e il numero del match (match_num) per mantenere la sequenza logica del tabellone.

Infine salviamo il risultato in un nuovo file CSV (atp_matches_2000_2024_raw.csv)

In [7]:
# Unione
df_total = pd.concat(li, axis=0, ignore_index=True)

# Ordinamento CRONOLOGICO (Fondamentale per l'ELO, anche se dovrebbero già essere ordinati)
df_total = df_total.sort_values(by=['tourney_date', 'tourney_id', 'match_num']).reset_index(drop=True)

#Salvo il dataset intero
df_total.to_csv(OUTPUT_FILENAME, index=False)
print(f"--- COMPLETATO ---")
print(f"File unito salvato come: {OUTPUT_FILENAME}")
print(f"Totale partite: {len(df_total)}")
print(df_total[['tourney_date', 'winner_name', 'loser_name']].head())

df_total.head()

--- COMPLETATO ---
File unito salvato come: atp_matches_2000_2024_raw.csv
Totale partite: 74906
  tourney_date          winner_name        loser_name
0   2000-01-03       Thomas Enqvist    Arnaud Clement
1   2000-01-03        Roger Federer  Jens Knippschild
2   2000-01-03  Jan Michael Gambill     Wayne Arthurs
3   2000-01-03   Sebastien Grosjean       Andrew Ilie
4   2000-01-03        Magnus Norman      Scott Draper


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2000-339,Adelaide,Hard,32,A,2000-01-03,1,102358,1.0,NaN,...,37.0,25.0,13.0,9.0,2.0,4.0,4.0,2606.0,56.0,805.0
1,2000-339,Adelaide,Hard,32,A,2000-01-03,2,103819,NaN,NaN,...,15.0,13.0,12.0,8.0,0.0,3.0,64.0,749.0,91.0,525.0
2,2000-339,Adelaide,Hard,32,A,2000-01-03,3,102998,NaN,NaN,...,59.0,49.0,22.0,16.0,4.0,5.0,58.0,803.0,105.0,449.0
3,2000-339,Adelaide,Hard,32,A,2000-01-03,4,103206,7.0,NaN,...,22.0,12.0,8.0,8.0,1.0,6.0,27.0,1298.0,54.0,845.0
4,2000-339,Adelaide,Hard,32,A,2000-01-03,5,102796,3.0,NaN,...,40.0,25.0,16.0,10.0,7.0,10.0,15.0,1748.0,154.0,297.0
